<a href="https://colab.research.google.com/github/jyotinayak321/IBM-ML-Training/blob/main/MICE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [19]:
#step1 load and pprpare data
df = np.round(pd.read_csv('50_Startups.csv')[
    ['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']
]/10000)

In [20]:
df

,R&D Spend,Administration,Marketing Spend,Profit
0,17.0,14.0,47.0,19.0
1,16.0,15.0,44.0,19.0
2,15.0,10.0,41.0,19.0
3,14.0,12.0,38.0,18.0
4,14.0,9.0,37.0,17.0
5,13.0,10.0,36.0,16.0
6,13.0,15.0,13.0,16.0
7,13.0,15.0,32.0,16.0
8,12.0,15.0,31.0,15.0
9,12.0,11.0,30.0,15.0


In [21]:
#sample 5 rows
np.random.seed(9)
df = df.sample(5)

In [22]:
df


,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [23]:
#replace target columns (we only impute input feeatures)
df = df.iloc[:,0:-1]


In [24]:
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [25]:
#step2:Introduce missing values
df.iloc[1, 0] = np.nan   #R&D Speed missing in row 37
df.iloc[3, 1] = np.nan   #Administration missing in row 14
df.iloc[-1, -1] = np.nan #Profit missing in last row


print("original data with missing VALUES")
print(df)

original data with missing VALUES
    R&D Spend  Administration  Marketing Spend
21        8.0            15.0             30.0
37        NaN             5.0             20.0
2        15.0            10.0             41.0
14       12.0             NaN             26.0
44        2.0            15.0              NaN


/tmp/ipykernel_590/3539506679.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1, 0] = np.nan   #R&D Speed missing in row 37
/tmp/ipykernel_590/3539506679.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3, 1] = np.nan   #Administration missing in row 14
/tmp/ipykernel_590/3539506679.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1, -1] = np.nan #Profit missing in last row


# Iteration 0 :mean Imputaion


In [26]:
#interaion with mean imputation
df0 = pd.DataFrame()
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

print("After Iteration 0 (mean imputation):",df0)


After Iteration 0 (mean imputation):     R&D Spend  Administration  Marketing Spend
21       8.00           15.00            30.00
37       9.25            5.00            20.00
2       15.00           10.00            41.00
14      12.00           11.25            26.00
44       2.00           15.00            29.25


# Iteration 1: First MICE Pass


In [27]:
from sklearn import linear_model
# columns1: Impute R&D Spend
df1 = df0.copy()
df1.iloc[1, 0] = np.nan

# create training data
X_train = df1.iloc[[0, 2, 3, 4], 1:3]
y_train = df1.iloc[[0, 2, 3, 4], 0]

#Train model
lr = LinearRegression()
lr.fit(X_train, y_train)

# predict missing value
X_test = df.iloc[1, 1:].values.reshape(1,2)
prediction = lr.predict(X_test)
df1.iloc[1, 0] = prediction

print(f"prediction R&D Spend for 37: {prediction[0]:.2f}")

#column 2:Impute Administration
df.iloc[3, 1] = np.nan  #restore nan

X_train = df1.iloc[[0, 1, 2, 4], 0:2]   #Cols 1 & 3
y_train = df1.iloc[[0, 1, 2, 4],1]      #Target:administration
lr = LinearRegression()
lr.fit(X_train, y_train)

X_test = df.iloc[3, [0,2]].values.reshape(1,2)
prediction = lr.predict(X_test)[0]
df1.iloc[3, 1] = prediction

print(f"prediction Administration for 14: {prediction:.2f}")

#Columns 3:imputer marketing spend
df1.iloc[4, -1]  = np.nan  #restore nan

X_train = df1.iloc[0:4, 0:2] #first 4 rows,cols 1&2
y_train = df1.iloc[0:4, 2]   #target:marketing spends
lr = LinearRegression()
lr.fit(X_train, y_train)

X_test = df.iloc[4, 0:2].values.reshape(1,2)
prediction = lr.predict(X_test)[0]
df1.iloc[4, 2] = prediction

print(f"prediction Marketing Spend for 1000000: {prediction:.2f}")

print("\nAfter Iteration 1 (First MICE Pass):")
print(df1)



prediction R&D Spend for 37: 23.14
prediction Administration for 14: 26.00
prediction Marketing Spend for 1000000: 42.05

After Iteration 1 (First MICE Pass):
    R&D Spend  Administration  Marketing Spend
21   8.000000            15.0        30.000000
37  23.141587             5.0        20.000000
2   15.000000            10.0        41.000000
14  12.000000            26.0        26.000000
44   2.000000            15.0        42.048836


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/tmp/ipykernel_590/748243147.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3, 1] = np.nan  #restore nan
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [28]:
#calculates difference
difference = df1 - df0
print("\nDifference:(Iteration0 + Iteration0)")
print(difference)


Difference:(Iteration0 + Iteration0)
    R&D Spend  Administration  Marketing Spend
21   0.000000            0.00         0.000000
37  13.891587            0.00         0.000000
2    0.000000            0.00         0.000000
14   0.000000           14.75         0.000000
44   0.000000            0.00        12.798836


# Complete the loop Implementation

In [144]:
import numpy as np
from sklearn.linear_model import LinearRegression


def mice_imputation(df_original, n_iterations=10, tolerance=1e-3):
    # tolerance : float, convergence threshold

    # Store which values were originally missing
    # Original missing values ka record pehle save karna hai
    missing_mask = df_original.isna()

    # Initialize with mean imputation
    df = df_original.copy()

    # Replace missing values with column mean
    for col in df_original.columns:
        # loop will apply on each col of the data set
        # all the changes are done in df, not in original dataset
        df[col] = df[col].fillna(df[col].mean())
        # nan value is replaced by the mean of the column

    # Store the current state
    df_current = df.copy()

    # Store history of each iteration
    history = [df_current.copy()]


    # Iteration loop
    for iteration in range(n_iterations):
        # iteration will run till n (10)

        # Save the dataframe of current iteration
        # Current iteration and previous iteration are saved for convergence
        df_previous = df_current.copy()


        # Loop through each column
        for col_idx, col_name in enumerate(df_current.columns):
            # Inside MICE, it will process each column one by one

            # Check if this column has missing values
            # Skip the column if the original column has no missing values
            if not missing_mask[col_name].any():
                continue


            # Restore the NaN for this column
            # Restore the originally missing values in this column as NaN
            # so that MICE can re-impute them using the other features
            df_current.loc[
                missing_mask[col_name], col_name
            ] = np.nan


            # Get indices for training and prediction

            # This finds the rows where the current column's value
            # was originally available (not missing)
            # These rows will be used to train the model
            train_idx = missing_mask[col_name].index[
                missing_mask[col_name] == False
            ]

            # This finds the rows where the current column's value
            # was originally missing
            # These rows will be used for prediction/imputation
            pred_idx = missing_mask[col_name].index[
                missing_mask[col_name] == True
            ]


            # Prepare Training Data

            # This creates a list of all columns except
            # the column currently being imputed
            other_cols = [
                c for c in df_current.columns
                if c != col_name
            ]

            # This creates the input features for training
            X_train = df_current.loc[train_idx, other_cols]

            # This creates the target/output for training
            y_train = df_current.loc[train_idx, col_name]


            # Train model
            model = LinearRegression()
            model.fit(X_train, y_train)


            # Impute/predict missing values
            X_pred = df_current.loc[pred_idx, other_cols]

            # Predict the missing values
            prediction = model.predict(X_pred)

            # Replace missing values with predicted values
            df_current.loc[pred_idx, col_name] = prediction


        # Store history
        history.append(df_current.copy())


        # Check the convergence
        difference = np.abs(df_current - df_previous)

        # Find the maximum change between current and previous iteration
        max_change = difference.max().max()

        print(
            f"Iteration {iteration + 1}: "
            f"Max change = {max_change:.6f}"
        )


        # If maximum change is smaller than tolerance,
        # the algorithm has converged
        if max_change < tolerance:
            print(
                f"Converged after {iteration + 1} iterations."
            )
            break


    return df_current, history

#RUN MICE
df_imputed, history = mice_imputation(df, n_iterations=15, tolerance=0.01)


print("\nFinal sataset")
print(df_imputed)


# Compare original and final MICE-imputed data
print("========================================")
print("Comparesion")

comparison = pd.DataFrame({
    "Original": df.loc[missing_mask.any(axis=1)].stack(),
    "MICE Imputed": df_imputed.loc[missing_mask.any(axis=1)].stack()
})

print(comparison)
print("========================================")
# Before and After MICE comparison

comparison_table = pd.DataFrame({
    "Before MICE": df_original.stack(),
    "After MICE": df_imputed.stack()
})

display(comparison_table)


Iteration 1: Max change = 13.891587
Iteration 2: Max change = 7.788589
Iteration 3: Max change = 23.948920
Iteration 4: Max change = 7.612848
Iteration 5: Max change = 0.272354
Iteration 6: Max change = 0.012787
Iteration 7: Max change = 0.000597
Converged after 7 iterations.

Final sataset
    R&D Spend  Administration  Marketing Spend
21   8.000000       15.000000        30.000000
37  26.718371        5.000000        20.000000
2   15.000000       10.000000        41.000000
14  12.000000       13.022364        26.000000
44   2.000000       15.000000        70.692041
Comparesion
                    Original  MICE Imputed
14 Administration        NaN     13.022364
   Marketing Spend      26.0     26.000000
   R&D Spend            12.0     12.000000
37 Administration        5.0      5.000000
   Marketing Spend      20.0     20.000000
   R&D Spend             NaN     26.718371
44 Administration       15.0     15.000000
   Marketing Spend       NaN     70.692041
   R&D Spend             2.

Before MICE  After MICE
2  Administration          10.0   10.000000
   Marketing Spend         41.0   41.000000
   R&D Spend               15.0   15.000000
14 Administration           NaN   13.022364
   Marketing Spend         26.0   26.000000
   R&D Spend               12.0   12.000000
21 Administration          15.0   15.000000
   Marketing Spend         30.0   30.000000
   R&D Spend                8.0    8.000000
37 Administration           5.0    5.000000
   Marketing Spend         20.0   20.000000
   R&D Spend                NaN   26.718371
44 Administration          15.0   15.000000
   Marketing Spend          NaN   70.692041
   R&D Spend                2.0    2.000000